# Оценка релевантности организаций запросам на Яндекс.Картах с помощью LLM-агента

<img src="https://sun9-65.userapi.com/impg/N4y2cxlL7PauAs82tBNFOUAiNctFICWDy4Mbiw/Jiz1fb7NLWU.jpg?size=1080x1080&quality=95&sign=df2786058624d9ccac3ede4d5d056e2f&type=album" width="500" height="500" />


## Описание и загрузка данных

Данные: https://disk.yandex.ru/d/6d5hFHvpAZjQdw

Ваша задача -- предсказать колонку relevance, используя все остальные данные об организации. Загрузим данные и посмотрим на них

In [ ]:
import requests

public_url = "https://disk.yandex.ru/d/6d5hFHvpAZjQdw"  # твоя публичная ссылка
api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"

resp = requests.get(api_url, params={"public_key": public_url})
resp.raise_for_status()
download_url = resp.json()["href"]  # это уже прямая ссылка на файл

dest = "/content/data.jsonl"
with requests.get(download_url, stream=True) as r:
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)


In [ ]:
import json
import pandas as pd

records = []
with open("/content/data.jsonl", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        if i == 2659:      # пропускаем битую строку
            continue
        try:
            obj = json.loads(line)
            records.append(obj)
        except Exception as e:
            print("ещё битая строка:", i, e)

data = pd.DataFrame(records)


In [ ]:
data['relevance'].unique()

array([1. , 0. , 0.1])

In [ ]:
data['relevance'].value_counts()

,count
relevance,
1.0,15881
0.0,14509
0.1,4703


Здесь 1.0 соответствует оценке RELEVANT_PLUS, 0.1 -- оценке RELEVANT_MINUS, 0.0 -- оценке IRRELEVANT.

Ваша задача -- построить LLM-агента, который будет предсказывать релевантность.

Выделим данные для оценки качества агента. Запуск агента -- это тяжелая и потенциально дорогая операция. Поэтому eval-множество имеет размер 500. Также для простоты из eval-множества выкинуты данные с оценкой RELEVANT_MINUS. Тем не менее, вы можете использовать такие примеры для подачи примеров агенту.

**ОБРАТИТЕ ВНИМАНИЕ, ЧТО В EVAL-ДАННЫЕ НЕЛЬЗЯ ПОДГЛЯДЫВАТЬ ДЛЯ КАЛИБРОВКИ АГЕНТА!!! ДЛЯ ЭТОГО ЕСТЬ ОБУЧАЮЩИЕ ДАННЫЕ**

В качестве метрики качества мы будем использовать обычную ACCURACY, поскольку классы сбалансированы.

In [ ]:
train_data = data[570:]
eval_data = data[:570]
eval_data = eval_data[eval_data["relevance"] != 0.1]
eval_data

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,relevance,reviews_summarized
0,сигары,"Москва, Дубравная улица, 34/29",Tabaccos; Магазин Tabaccos; Табаккос,Магазин табака и курительных принадлежностей,1263329400,None,1.0,"Организация занимается продажей табака, курите..."
1,кальянная спб мероприятия,"Санкт-Петербург, Большой проспект Петроградско...",PioNero; Pionero; Пицца Паста бар; Pio Nero; P...,Кафе,228111266197,PioNero предлагает разнообразные блюда итальян...,0.0,"Организация PioNero — это кафе, бар и ресторан..."
2,Эпиляция,"Московская область, Одинцово, улица Маршала Жу...",MaxiLife; Центр красоты и здоровья MaxiLife; Ц...,Стоматологическая клиника,1247255817,"Стоматологическая клиника, массажный салон и к...",1.0,"Организация занимается стоматологическими, кос..."
4,стиральных машин,"Москва, улица Обручева, 34/63",М.Видео; M Video; M. Видео; M.Видео; Mvideo; М...,Магазин бытовой техники,1074529324,М.Видео предлагает широкий ассортимент бытовой...,1.0,Организация занимается продажей бытовой техник...
5,сеть быстрого питания,"Санкт-Петербург, 1-я Красноармейская улица, 15",Rostic's; KFC; Ресторан быстрого питания KFC,Быстрое питание,1219173871,Rostic's предлагает различные наборы быстрого ...,1.0,"Организация занимается быстрым питанием, предо..."
...,...,...,...,...,...,...,...,...
561,наращивание ресниц,"Саратов, улица имени А.С. Пушкина, 1",Сила; Sila; Beauty brow; Студия бровей Beauty ...,Салон красоты,236976975812,Салон красоты «Сила» предлагает услуги по уход...,1.0,Организация «Сила» занимается предоставлением ...
565,игры,"Москва, Щёлковское шоссе, 79, корп. 1",YouPlay; YouPlay КиберКлуб,Компьютерный клуб,109673025161,YouPlay КиберКлуб предлагает услуги по игре на...,0.0,Организация занимается предоставлением услуг к...
566,домашний интернет в курске что подключить отзы...,"Курск, Садовая улица, 5",Цифровой канал; Digital Channel; DChannel; ЦК;...,Телекоммуникационная компания,1737991898,None,0.0,None
567,гостиница волгодонск сауна номер телефона,"Ростовская область, городской округ Волгодонск...",Поплавок; Poplavok,"База , дом отдыха",147783493467,"Предлагает размещение в различных типах жилья,...",0.0,Организация «Поплавок» предлагает услуги базы ...


In [ ]:
eval_data.to_excel("eval_data.xlsx")

## Импорты

In [ ]:
!pip install -q langchain langgraph langchain_openai langchain_core langchain_community

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.tools import TavilySearchResults
from langchain_core.tools import tool

import json
import re
from typing import TypedDict, Dict, Any, List, Optional, Literal
import os
import time, uuid
import math
from dotenv import load_dotenv

In [ ]:
load_dotenv()

True

# Декомпозированный код в LangGraph

In [ ]:
load_dotenv()

True

In [ ]:
import os, re, math, json, random
from typing import Any, Dict, List, Optional, TypedDict, Literal

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage,
)

MODEL_NAME = "arcee-ai/trinity-large-preview:free"

TAVILY_MAX_RESULTS = 20
TAVILY_SEARCH_DEPTH = "advanced"

MAX_SEARCH_CALLS = 3            # жёсткий лимит на поиск
TEMPERATURE = 0.4
TIMEOUT_S = 90
MAX_RETRIES = 3
MAX_TURNS = 12                  # страховка от бесконечного цикла


def _clean_str(x) -> str:
    if x is None:
        return ""
    try:
        if isinstance(x, float) and math.isnan(x):
            return ""
    except Exception:
        pass
    s = str(x).strip()
    return "" if s.lower() in {"nan", "none"} else s

def make_card_text(row: Dict[str, Any], max_reviews_len: int = 1500) -> str:
    """Собираем card_text из полей карточки (без label/relevance)."""
    name = _clean_str(row.get("name", ""))
    address = _clean_str(row.get("address", ""))
    rubric = _clean_str(row.get("normalized_main_rubric_name_ru", row.get("rubric", "")))
    permalink = _clean_str(row.get("permalink", ""))
    prices = _clean_str(row.get("prices_summarized", ""))
    reviews = _clean_str(row.get("reviews_summarized", ""))

    lines = ["Информация об организации:"]
    if name: lines.append(f"- Названия (через ;): {name}")
    if rubric: lines.append(f"- Рубрика: {rubric}")
    if address: lines.append(f"- Адрес: {address}")
    if permalink: lines.append(f"- Permalink: {permalink}")
    if prices: lines.append(f"- Услуги/цены (summary): {prices}")
    if reviews: lines.append(f"- Отзывы (summary): {reviews[:max_reviews_len]}")
    return "\n".join(lines).strip()

def parse_final_label(text: str) -> Optional[int]:
    """Принимаем финал только если это '0' или '1' иначе возвращаем None"""
    if text is None:
        return None
    s = str(text).strip()
    return int(s) if s in {"0", "1"} else None

def normalize_search_query(q: str) -> str:
    """Нормализация для анти-дубликатов."""
    s = (q or "").lower().strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\sа-яё]", " ", s, flags=re.IGNORECASE)
    s = re.sub(r"\s+", " ", s).strip()
    return s


In [ ]:
from langgraph.graph import StateGraph, END

class AgentState(TypedDict, total=False):
    # входные данные
    query: str
    card_text: str
    org_name: str
    org_address: str

    # “память” агента
    messages: List[Any]
    search_queries: List[str]
    search_calls: int
    last_tool_result: Optional[str]     # сырой текст результата последней тулзы

    # управление циклом/выход
    final_label: Optional[int]          # 0/1 когда принято решение
    stop_reason: Optional[str]          # для дебага
    turn: int                           # сколько узлов-итераций прошло

    # параметры веб-поиска
    fact_x: Optional[str]
    must_keywords: List[str]     # нормализованный список ключевых слов (строки)
    must_stems: List[str]

tavily = TavilySearchResults(
    max_results=TAVILY_MAX_RESULTS,  # 5–10 обычно хватает
    search_depth="basic",
    include_answer=False,
    include_raw_content=False,       # КРИТИЧНО: raw_content может быть огромным
    include_images=False,
    tavuly_api_key=os.environ["TAVILY_API_KEY"]
)


llm = ChatOpenAI(
    model=MODEL_NAME,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=TEMPERATURE,
    max_tokens=8000,
    timeout=TIMEOUT_S,
    max_retries=MAX_RETRIES,
).bind_tools([tavily])

SYSTEM_PROMPT = """
# Роль:
Ты — бинарный классификатор релевантности организации рубричному запросу.
Твоя задача — определить, релевантен ли запрос конкретной организации.

Тебе даны:
- query: запрос пользователя
- card_text: карточка организации (описание, рубрика, услуги, цены, отзывы)

Ты можешь использовать инструмент веб-поиска:
- tavily_search_results_json

# Правила:
## Принципы принятия решения:
1. Очевидное несоответствие:
   Если по card_text видно, что организация точно НЕ может соответствовать запросу
   (другая сфера/рубрика, другой тип заведения), то верни 0 без веб-поиска.

2. Ответ = 1 только если есть ЯВНОЕ подтверждение соответствия запросу (query).
   Подтверждение должно быть:
   - либо в карточке организации (card_text),
   - либо в результатах веб-поиска, и относиться именно к этой организации
     (совпадает название, адрес или официальный домен).

3. Если запрос содержит обязательные ограничения,
   Например: (район, адрес, режим работы, конкретная услуга, год и т.п.),
   то ответ = 1 только если эти ограничения ЯВНО подтверждены.
   Если обязательное ограничение

4. Если запрос содержит субъективные характеристики
   (уютный, романтичный, недорого, лучший и т.п.),
   то ответ = 1 только если в отзывах есть упоминания соответствующего качества.
   Если таких упоминаний нет — ответ = 0.

5. Если запрос по смыслу полностью совпадает с основной рубрикой организации
   и не содержит дополнительных ограничений, то ответ = 1.

6. Если в карточке организации (card_text) есть необходимая информация,
   которая полностью подтверждает все детали запроса, то ответ = 1 без веб-поиска.

7. Если в карточке организации (card_text) нет необходимой информация,
   которая подтверждает все детали запроса, то веб-поиск необходим.

## Правила веб-поиска:
0. Если информации в карточке организации (card_text) достаточно для принятия решения,
   то веб-поиск НЕЛЬЗЯ использовать.

1. Используй инструмент tavily_search_results_json,
   ТОЛЬКО ЕСЛИ информации в card_text недостаточно для принятия решения.

2. Цель веб-запроса: подтвердить fact_x.
   В веб-запрос ОБЯЗАТЕЛЬНО включай:
   - название организации,
   - адрес или город,
   - и конкретную деталь запроса, которую нужно подтвердить.

3. Максимум 3 веб-запроса.
   После 3 запросов ты ОБЯЗАН принять финальное решение.

4. Если результаты веб-запроса не подтвердили и не опровергли конкретную деталь запроса,
   то необходимо сформулировать новый веб-запрос, который должен быть принципиально другим:
   - другая формулировка проверяемого факта,
   - или использование синонимов
   - или использование других ключевых слов, связанных с этой деталью.

5. Если 2 веб-запроса не подтвердили и не опровергли конкретную деталь запроса,
   то необходимо сделать back step :
   Цель back step — расширить поиск, чтобы найти более общий источник информации,
   внутри которого может содержаться искомый факт.

   - Не пытайся искать точную фразу запроса "слово в слово".
   - Перейди к более общему контексту, в котором искомый факт
     естественно может быть упомянут.

   Это означает:
   - искать разделы сайта (меню, прайс, услуги, описание, отзывы),
   - искать категорию услуг, внутри которой может находиться нужная деталь,
   - искать списки или перечни, где эта деталь может быть одной из позиций.

   Пример back step:
  - вместо запроса "салаты с огурцами и помидорами в масле в ресторане <название ресторана> <город>"
  сделать back step: "<название ресторана> <город> меню ресторана".

  Back step выполняется ТОЛЬКО после двух неудачных попыток поиска конкретной детали.

6. Если после 3 веб-запросов явного подтверждения fact_x нет — считай, что факт НЕ подтверждён => ответ = 0.

7. Результат веб-поиска может подтвердить или опровергнуть fact_x ТОЛЬКО в том случае,
   если он относится к этой организации (совпадает название или адрес).
   Если результат веб-поиска не связан с организацией из card_text —
   то эту информацию использовать НЕЛЬЗЯ.

## Перед первым веб-поиском:
ОЧЕНЬ ВАЖНО:
Если ты ещё не выбрал fact_x и must_keyword, то НЕ вызывай веб-поиск.
Сначала верни СТРОГО в JSON формате с такими полями:
{"fact_x": "...", "must_keyword": "..."}

Требования к fact_x и must_keyword:
- fact_x: одно короткое конкретное предложение — какой факт нужно подтвердить/опровергнуть.
- must_keyword: одно (максимум два если они зваимозаменяемы, но лучше одно) слово, которое ОБЯЗАНО встречаться на релевантной веб-странице,
  если факт действительно подтверждается. Слово должно быть достаточно общим.
### Примеры выбора fact_x и must_keyword по query (формат ответа — строго JSON в одну строку):
1) query: "медсправка на права".
  {"fact_x": "Организация <название организации> оформляет медицинскую справку для водителей.",
  "must_keyword": "справка"}
2) query: "танцы для детей 1.5 лет"
{"fact_x": "Организация <название организации> проводит занятия танцами для детей (в том числе для детей возраста 1.5 года).",
"must_keyword": "танец"}
3) query: "ударно-волновая терапия в екатеринбурге"
{"fact_x": "Организация <название организации> оказывает услугу ударно-волновой терапии (УВТ).",
"must_keyword": "терапия"}
4) query: "круглосуточная аптека"
{"fact_x": "Аптека работает круглосуточно (24/7).", "must_keyword": ["круглосуточно", "24"]}

После этого (на следующих шагах) ты можешь вызывать веб-поиск или давать финальный ответ 0/1.
JSON должен быть единственным содержимым сообщения (без текста вокруг).

## Обязательное действие:
На каждом шаге ты обязан:
- либо формулируешь fact_x и must_keyword в формате JSON,
- либо вызвать инструмент веб-поиска для подтверждения fact_x
  (если fact_x и must_keyword уже есть),
- либо выдать финальный ответ 0 или 1.

Ответ без вызова инструмента и без финальной классификации недопустим.

## Формат финального ответа:
Только когда принято окончательное решение, то нужно вернуть РОВНО один символ:
0 или 1
Без пояснений, пробелов и дополнительных символов.
"""



/tmp/ipython-input-3146922301.py:26: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults(


In [ ]:
pip install pymystem3

In [ ]:
import re
from typing import Any, Dict, List, Optional
import nltk
from nltk.stem import SnowballStemmer
nltk.download('punkt')
from pymystem3 import Mystem
lemmatizer = Mystem()
bans_lemmas = [' ', '\n']

def _strip_html(text: str) -> str:
    # очень простой HTML-stripper (достаточно для Tavily content)
    text = re.sub(r"<script\b[^>]*>.*?</script>", " ", text, flags=re.I | re.S)
    text = re.sub(r"<style\b[^>]*>.*?</style>", " ", text, flags=re.I | re.S)
    text = re.sub(r"<[^>]+>", " ", text)
    return text

def _normalize_ws(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def _safe_str(x: Any) -> str:
    if x is None:
        return ""
    return str(x)

def _extract_best_text(d: Dict[str, Any]) -> str:
    """
    Tavily может возвращать поля: content, snippet, raw_content, body и т.п.
    Берём самое полезное по приоритету.
    """
    for key in ("content", "snippet", "raw_content", "body", "text"):
        if key in d and d[key]:
            return _safe_str(d[key])
    return ""

def _smart_clip(text: str, max_chars: int) -> str:
    """
    Обрезаем умнее, чем просто text[:max_chars]:
    - берём начало + кусок из середины (часто там "услуги/цены/контакты")
    """
    text = text.strip()
    if len(text) <= max_chars:
        return text

    head = int(max_chars * 0.6)
    tail = max_chars - head - 60  # место под маркер
    if tail < 200:
        tail = 200
        head = max_chars - tail - 60

    mid_start = max(0, (len(text) // 2) - (tail // 2))
    mid = text[mid_start: mid_start + tail]
    return text[:head].rstrip() + "\n…[snip]…\n" + mid.strip()

def lemmatize_text(text: str) -> list:
    """Лемматизируем текст и возвращаем список лемм."""
    return [lemma for lemma in lemmatizer.lemmatize(text.lower()) if lemma not in bans_lemmas]

def process_tavily_result(
    tavily_result: Any,
    *,
    max_total_chars: int = 6000,
    max_source_chars: int = 900,
    min_source_chars: int = 120,
    max_sources: int = 12,
    state: AgentState,
) -> str:
    """
    Превращает выдачу Tavily в компактный readable blob для LLM.

    - max_total_chars: общий лимит текста, который пойдёт в контекст модели
    - max_source_chars: лимит на один источник (после чистки)
    - min_source_chars: слишком короткие выкидываем (обычно мусор)
    - max_sources: ограничиваем число источников
    """
    must_keywords = state.get('must_keywords', [])
    lemmatized_keywords = lemmatize_text(" ".join(must_keywords))

    # 1) Приводим к списку dict
    items: List[Dict[str, Any]] = []
    if isinstance(tavily_result, str):
        # иногда Tavily может вернуть строку — тогда просто клипнем
        txt = _normalize_ws(_strip_html(tavily_result))
        return _smart_clip(txt, max_total_chars)

    if isinstance(tavily_result, dict):
        # иногда это {"results":[...]} или просто один result
        if "results" in tavily_result and isinstance(tavily_result["results"], list):
            raw = tavily_result["results"]
        else:
            raw = [tavily_result]
    elif isinstance(tavily_result, list):
        raw = tavily_result
    else:
        raw = [{"content": _safe_str(tavily_result)}]

    for it in raw:
        if isinstance(it, dict):
            items.append(it)
        else:
            items.append({"content": _safe_str(it)})

    # 2) Нормализуем + чистим + режем каждый источник
    seen_urls = set()
    blocks: List[str] = []
    total = 0

    for it in items:
        title = _safe_str(it.get("title") or it.get("name") or "").strip()
        url = _safe_str(it.get("url") or it.get("link") or it.get("source") or "").strip()

        # дедуп по URL (если есть)
        if url:
            ukey = url.lower()
            if ukey in seen_urls:
                continue
            seen_urls.add(ukey)

        text = _extract_best_text(it)
        text = _strip_html(text)
        text = _normalize_ws(text)

        # отрезаем “шум”: слишком короткие куски часто бесполезны
        if len(text) < min_source_chars:
            continue

        text = _smart_clip(text, max_source_chars)

        header = []
        if title:
            header.append(f"TITLE: {title}")

        block = ""
        if header:
            block += "\n".join(header) + "\n"
        block += "TEXT:\n" + text

        # ограничиваем количество источников
        if len(blocks) >= max_sources:
            break

        # ограничиваем общий размер
        if total + len(block) + 40 > max_total_chars:
            remaining = max_total_chars - total - 40
            if remaining <= 200:
                break
            # обрежем block до remaining
            block = _smart_clip(block, remaining)

        lemmatized_page_text = [lemma for part in (text, title) for lemma in lemmatize_text(part) ]
        if any(lemma in lemmatized_page_text for lemma in lemmatized_keywords):
          blocks.append(block)
          total += len(block) + 40


        if total >= max_total_chars:
            break

    if not blocks:
        return ""
    return ("\n\n<NEXT_SOURCE>\n\n").join(blocks)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
Installing mystem to /root/.local/bin/mystem from http://download.cdn.yandex.net/mystem/mystem-3.1-linux-64bit.tar.gz


In [ ]:
def parse_plan_json(text: str) -> Optional[Dict[str, Any]]:
    """Парсит fact_x, must_keywords JSON
    Если получилось распарсить, возвращает словарь
    Иначе возвращает None """
    if not text:
        return None
    s = str(text).strip()

    # часто модель может вернуть ```json ... ```
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s.strip(), flags=re.I)
        s = re.sub(r"\s*```$", "", s.strip())

    if not (s.startswith("{") and s.endswith("}")):
        return None

    try:
        obj = json.loads(s)
    except Exception:
        return None

    fact_x = str(obj.get("fact_x", "")).strip()
    mk = obj.get("must_keyword", None)

    if not fact_x or mk is None:
        return None

    # mk может быть строкой или списком строк
    if isinstance(mk, str):
        kws = [mk.strip()]
    elif isinstance(mk, list):
        kws = [str(x).strip() for x in mk if str(x).strip()]
    else:
        return None

    kws = [k for k in kws if k]
    if not kws:
        return None

    return {"fact_x": fact_x, "must_keywords": kws}


In [ ]:
def build_initial_state(row: Dict[str, Any]) -> AgentState:
    query = _clean_str(row.get("Text", ""))
    card_text = make_card_text(row, max_reviews_len=1500)

    org_name = _clean_str(row.get("name", ""))
    org_address = _clean_str(row.get("address", ""))

    messages: List[Any] = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"query: {query}\n\ncard_text:\n{card_text}")
    ]

    return AgentState(
        query=query,
        card_text=card_text,
        org_name=org_name,
        org_address=org_address,
        messages=messages,
        search_queries=[],
        search_calls=0,
        last_tool_result=None,
        final_label=None,
        stop_reason=None,
        turn=0,
    )

def is_near_duplicate(new_q: str, prev_qs: List[str]) -> bool:
    """Простой анти-дубликат: сравниваем нормализованные строки."""
    n = normalize_search_query(new_q)
    if not n:
        return True
    prev_norm = [normalize_search_query(x) for x in prev_qs]
    return n in prev_norm

def agent_step(state: AgentState) -> AgentState:
    """
    Один шаг LLM.
    - Если модель дала финальный ответ (строго 0/1) -> final_label
    - Если модель запросила tool_calls -> оставляем в messages и пойдём в tool_step
    - Если модель выдала мусор/пустоту -> делаем repair-turn (просим ровно 0/1 или tool)
    """
    state["turn"] = int(state.get("turn", 0)) + 1

    # Если уже есть final_label — ничего не делаем
    if state.get("final_label") in (0, 1):
        state["stop_reason"] = state.get("stop_reason") or "already_final"
        return state

    resp = llm.invoke(state["messages"])
    # ---- ГЕЙТ: план до первого веб-поиска ----
    if not state.get("fact_x"):
        plan = parse_plan_json(getattr(resp, "content", None))
        # если модель попыталась tool_calls до планирования — блокируем
        if getattr(resp, "tool_calls", None) and not plan:
            state["messages"].append(
                HumanMessage(
                    content=(
                        "Сначала выбери fact_x и must_keyword. "
                        "Верни ТОЛЬКО JSON: {\"fact_x\": \"...\", \"must_keyword\": \"...\"} "
                        "или {\"fact_x\": \"...\", \"must_keyword\": [\"...\", \"...\"]}. "
                        "НЕ вызывай веб-поиск в этом сообщении."
                    )
                )
            )
            state["stop_reason"] = "tool_blocked_until_plan"
            return state
        # если план распарсился — сохраняем и продолжаем цикл (на следующем шаге модель сможет искать)
        if plan:
            state["fact_x"] = plan["fact_x"]
            state["must_keywords"] = plan["must_keywords"]
            state["stop_reason"] = "plan_ok"
            # Подсказка модели, что план принят и можно искать
            state["messages"].append(
                HumanMessage(
                    content=(
                        f"fact_x и must_keyword установлены.\n"
                        f"fact_x: {state['fact_x']}\n"
                        f"must_keyword: {state['must_keywords']}\n"
                        "Теперь либо вызови веб-поиск для подтверждения fact_x, "
                        "либо дай финальный ответ 0/1 (если уже достаточно card_text)."
                    )
                )
            )
            return state

        # если не план и не tool_calls — просим строго JSON
        state["messages"].append(
            HumanMessage(
                content=(
                    "Нужно сначала выбрать fact_x и must_keyword. "
                    "Верни ТОЛЬКО JSON вида "
                    "{\"fact_x\": \"...\", \"must_keyword\": \"...\"} "
                    "или {\"fact_x\": \"...\", \"must_keyword\": [\"...\", \"...\"]}. "
                    "Без текста вокруг."
                )
            )
        )
        state["stop_reason"] = "plan_repair_requested"
        return state




    state["messages"].append(resp)

    # 1) Попытка считать финал
    label = parse_final_label(getattr(resp, "content", None))
    if label is not None:
        state["final_label"] = label
        state["stop_reason"] = "model_final"
        return state

    # 2) Если модель не просит тулзу и не дала финал — repair-turn
    tool_calls = getattr(resp, "tool_calls", None)
    if not tool_calls:
        # просим строго: либо tool-call, либо 0/1
        repair = HumanMessage(
            content=(
                "Формат нарушен. Сейчас сделай ОДНО из двух:\n"
                "1) ЛИБО вызови tavily_search_results_json с аргументом {\"query\": \"...\"}\n"
                "2) ЛИБО верни ровно один символ: 0 или 1.\n"
                "Никакого текста кроме tool-call или 0 / 1 ."
            )
        )
        state["messages"].append(repair)

        resp2 = llm.invoke(state["messages"])
        state["messages"].append(resp2)

        label2 = parse_final_label(getattr(resp2, "content", None))
        if label2 is not None:
            state["final_label"] = label2
            state["stop_reason"] = "repair_final"
            return state

        # если снова нет tool_calls — выходим в 0 (безопасный дефолт)
        if not getattr(resp2, "tool_calls", None):
            state["final_label"] = 0
            state["stop_reason"] = "repair_failed_default_0"
        return state

    # 3) tool_calls есть — идём в tool_step (ветвление на уровне графа)
    state["stop_reason"] = "tool_requested"
    return state

def tool_step(state: AgentState) -> AgentState:
    """
    Выполняем tool_calls последнего AIMessage.
    Enforcement:
    - максимум MAX_SEARCH_CALLS
    - анти-дубликаты
    - если лимит исчерпан -> просим финал без вызова тулзов
    """
    if state.get("final_label") in (0, 1):
        return state

    # Найдём последний AIMessage (он последний в messages после agent_step)
    last_msg = state["messages"][-1]
    tool_calls = getattr(last_msg, "tool_calls", None) or []

    # Если модель попросила тулзу, но лимит уже исчерпан — запрещаем тулзу и просим финал
    if int(state.get("search_calls", 0)) >= MAX_SEARCH_CALLS:
        state["messages"].append(
            HumanMessage(
                content=(
                    f"Лимит веб-поиска исчерпан (MAX={MAX_SEARCH_CALLS}). "
                    "Ты НЕ МОЖЕШЬ делать новые поисковые запросы. "
                    "Прими финальное решение и верни ровно один символ: 0 или 1."
                )
            )
        )
        # Вызовем модель ещё раз (tools связаны, но модель должна подчиниться лимиту)
        resp = llm.invoke(state["messages"])
        state["messages"].append(resp)
        label = parse_final_label(getattr(resp, "content", None))
        state["final_label"] = label if label is not None else 0
        state["stop_reason"] = "search_limit_forced_final"
        return state

    # Выполняем все tool_calls (но реально ожидаем tavily_search_results_json)
    tools_by_name = {t.name: t for t in [tavily]}

    for call in tool_calls:
        tool_name = call.get("name")
        tool_args = call.get("args") or {}
        tool_call_id = call.get("id")

        if tool_name != "tavily_search_results_json":
            # неизвестная тулза — просто игнорируем и продолжаем
            continue

        q = str(tool_args.get("query", "") or "").strip()
        prev_qs = state.get("search_queries", []) or []

        # анти-дубликат: если дубликат, просим другой запрос (без выполнения тулзы)
        if is_near_duplicate(q, prev_qs):
            state["messages"].append(
                HumanMessage(
                    content=(
                        "Этот веб-запрос слишком похож на уже сделанные. "
                        "Сформулируй ПРИНЦИПИАЛЬНО ДРУГОЙ запрос (другая стратегия/тип страницы/синонимы). "
                        "Либо, если доказательств уже достаточно — верни 0 или 1."
                    )
                )
            )
            # модель попробует снова
            resp = llm.invoke(state["messages"])
            state["messages"].append(resp)
            state["stop_reason"] = "duplicate_blocked"
            return state

        # выполняем тулзу
        tool = tools_by_name[tool_name]

        if tool_name == "tavily_search_results_json":
            raw = tool.invoke({"query": q})
            tool_result = process_tavily_result(raw, max_total_chars=6000, max_source_chars=900)
        else:
            tool_result = None

        state["search_queries"] = prev_qs + [q]
        state["search_calls"] = int(state.get("search_calls", 0)) + 1
        state["last_tool_result"] = tool_result if isinstance(tool_result, str) else str(tool_result)

        state["messages"].append(
            ToolMessage(
                content=state["last_tool_result"],
                tool_call_id=tool_call_id
            )
        )

        # после каждого поиска — напоминание, что либо новый принципиально другой поиск, либо финал
        state["messages"].append(
            HumanMessage(
                content=(
                    "Оцени результаты поиска.\n"
                    f"Сделано поисков: {state['search_calls']} из {MAX_SEARCH_CALLS}.\n"
                    "Если искомый факт подтверждён/опровергнут — верни 0 или 1.\n"
                    "Если нет — сформулируй следующий запрос принципиально иначе (см. правила), "
                    "а после 2 неудачных попыток сделай back step.\n"
                    "Финальный формат: только '0' или '1'."
                )
            )
        )

        # Обычно модель делает один tool-call за шаг, выходим — дальше граф снова вызовет agent_step
        state["stop_reason"] = "tool_executed"
        return state

    # Если tool_calls не было выполнено — вернёмся в агент
    state["stop_reason"] = "no_valid_tool_calls"
    return state


In [ ]:
def route_after_agent(state: AgentState) -> str:
    """Маршрутизация после agent_step: END если есть финал, иначе в tool_step."""
    if state.get("final_label") in (0, 1):
        return "end"
    # если последний шаг не привёл к финалу, но tool_calls могли быть — идём в tool_step
    # (tool_step сам проверит лимиты)
    return "tool"

def route_after_tool(state: AgentState) -> str:
    """После tool_step всегда возвращаемся к agent_step, если ещё нет финала."""
    if state.get("final_label") in (0, 1):
        return "end"
    # страховка от бесконечности
    if int(state.get("turn", 0)) >= MAX_TURNS:
        state["final_label"] = 0
        state["stop_reason"] = "max_turns_default_0"
        return "end"
    return "agent"

graph = StateGraph(AgentState)

graph.add_node("agent", agent_step)
graph.add_node("tool", tool_step)

graph.set_entry_point("agent")

graph.add_conditional_edges(
    "agent",
    route_after_agent,
    {
        "tool": "tool",
        "end": END,
    },
)

graph.add_conditional_edges(
    "tool",
    route_after_tool,
    {
        "agent": "agent",
        "end": END,
    },
)

app = graph.compile()

def langgraph_predict(row: Dict[str, Any],
                      debug: bool = False,
                      return_messages: bool = False,) -> int:
    state = build_initial_state(row)
    final_state = app.invoke(state)

    label = final_state.get("final_label")
    if label not in (0, 1):
        label = 0

    if debug:
        print("final_label:", label)
        print("stop_reason:", final_state.get("stop_reason"))
        print("search_calls:", final_state.get("search_calls"))
        print("search_queries:", final_state.get("search_queries"))
        # последний ответ модели
        last_ai = None
        for m in reversed(final_state.get("messages", [])):
            if isinstance(m, AIMessage):
                last_ai = m
                break
        if last_ai is not None:
            print("last_ai.content:", (last_ai.content or "")[:500])
        return int(label), final_state.get("messages")

    return int(label)


In [ ]:
label, messages = langgraph_predict(row = train_data.loc[1127],debug = True, return_messages=True)

final_label: 0
stop_reason: search_limit_forced_final
search_calls: 3
search_queries: ['Интердентос Пушкино зубов отель', 'Интердентос зубов отель Пушкино услуги', 'Интердентос Пушкино меню услуг цены']
last_ai.content: 0
